# Clean `emr_cycle_event`

A first-pass cleaning of the `emr_cycle_event` table (20,826 rows, 5 columns) before analysis.

What this notebook does, in order:
1. **Profile** every column (how full it is, how many distinct values, its type)
2. **Drop** rows that are not for IVF/Egg Retrievals
3. **Review** columns that hold only a single value
4. **Parse** the date columns (currently stored as text)
5. **Tidy** column types
6. **Save** a cleaned copy + keep an audit note of what changed

The same `profile()` function works on the other `emr_*` tables too, so you can reuse this pattern.

> **Before running:** finish the venv setup in this repo and install the packages — open a terminal and run `pip install pandas pyarrow`. The `requirements.txt` alongside this notebook lists everything.
>
> **Important:** keep your raw patient data in a `data/` folder that is git-ignored. Don't commit EMR exports to GitHub, even a private repo.

## Setup

In [1]:
# Importing necessary libraries
import pandas as pd

In [4]:
df = pd.read_csv(r"C:\Users\tamar.schaap\OneDrive - RPSD\project documents and materials\Datasets - nAble - For Tamar - UCSD - Confidential PHI\Datasets\emr_cycle_event.csv")
display(df.head())

print(df.shape)


,id,cycleid,label,daynum,eventdate
0,19722eb8-db05-4293-b099-e3a6b8059f09,efc0fffc-3612-4886-beee-536706179bce,LMP,1,2023-10-07
1,972f8f6f-bbd4-4960-ba6b-adec537b9912,8aa7163f-e300-41d4-b8cc-d12c3ee4f70c,XX,1,2023-12-17
2,f1806802-e10e-48db-98aa-18ec366fff78,c6e70948-17fb-4594-a3c3-657bff9f9557,STIM,1,2023-10-05
3,b0696bdb-5965-419d-813e-fe4e1c9f18f9,c6e70948-17fb-4594-a3c3-657bff9f9557,LMP,-32,2023-09-03
4,22e4cb56-84eb-4d2b-ba66-93fe7500ef77,2e42dd6e-ac6e-410d-920a-605e9b24a2a7,LMP,1,2023-07-15


(20826, 5)


In [5]:
# --- will be reusing these columns a lot for processing ---
ID_COL     = "cycleid"   # the column that groups rows into one cycle
LABEL_COL  = "label"     # the column holding LMP / STIM / IUI / ...
DAYNUM_COL = "daynum"    # day of cycle number (varies based on what kind of cycle/treatment)

## 1. Profile every column

One row per column so you can see at a glance what's worth keeping.

In [7]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: how full it is, how many distinct values, its dtype,
    and whether all non-null values are identical."""
    out = pd.DataFrame({
        "non_null": frame.notna().sum(),
        "nulls": frame.isna().sum(),
        "distinct": frame.nunique(dropna=True),
        "dtype": frame.dtypes.astype(str),
    })

    out["pct_null"] = (out["nulls"] / len(frame) * 100).round(1)

    # True if all non-null values in the column are the same
    out["all_values_same"] = out["distinct"] <= 1

    return out.sort_values("non_null", ascending=False)

In [8]:
prof = profile(df)
prof

,non_null,nulls,distinct,dtype,pct_null,all_values_same
id,20826,0,20826,str,0.0,False
cycleid,20826,0,9215,str,0.0,False
label,20826,0,22,str,0.0,False
daynum,20826,0,146,int64,0.0,False
eventdate,20826,0,1768,str,0.0,False


In [30]:
# list all distinct values of label column
print(df["label"].unique())

# now a count of each label
print(df[LABEL_COL].value_counts())

# LMP: Last menstrual period
# XX: 
# STIM: Ovarian stimulation
# IUI: Intrauterine insemination
# TRG: Trigger
# RET: Egg retrieval
# +LH: positive LH
# FET: Frozen embryo transfer
# TIC: Timed intercourse
# ET: Embryo transfer (Fresh)
# AFC: Antral follicle count
# CC:
# LH:
# THW: Thaw
# TEG:
# PG: 

<StringArray>
[       'LMP',         'XX',       'STIM',        'IUI',        'TRG',
        'RET',        '+LH',        'FET',        'TIC',         'ET',
        'AFC',         'CC',         'LH',        'THW',        'TEG',
         'PG', 'Day 12 Day', 'Day 11 TRG', 'Day 12 RET',  'Day 8 TRG',
 'Day 13 Day', 'STIM Day 1']
Length: 22, dtype: str
label
LMP           6334
TRG           3837
STIM          3618
RET           3053
FET           1456
+LH           1207
IUI            887
ET             147
XX             105
LH              74
TEG             55
Day 11 TRG      16
TIC             14
THW              9
CC               3
PG               3
Day 12 Day       3
AFC              1
Day 12 RET       1
Day 8 TRG        1
Day 13 Day       1
STIM Day 1       1
Name: count, dtype: int64


## 2. Creating dfs for retrievals and transfers

For now, this means getting rid of cycleid's that have specific rows with events not pertaining to those.

In [43]:
# ----------------------------
# Retrieval cycles
# ----------------------------
RETRIEVAL_DROP_LABELS = ["IUI", "TIC", "FET", "ET"]

labels = df[LABEL_COL].astype(str).str.strip().str.upper()

# Cycles to exclude
retrieval_drop_cycles = set(df.loc[labels.isin(RETRIEVAL_DROP_LABELS), ID_COL])

# Exclude cycles with LMP on day 1
lmp_day1_mask = (labels == "LMP") & (df[DAYNUM_COL] == 1)
retrieval_drop_cycles |= set(df.loc[lmp_day1_mask, ID_COL])

# Keep cycles that contain STIM but are not excluded
retrieval_cycles = set(df.loc[labels == "STIM", ID_COL])
retrieval_keep_ids = retrieval_cycles - retrieval_drop_cycles

retrievals = df[df[ID_COL].isin(retrieval_keep_ids)].copy()


# ----------------------------
# Transfer cycles
# ----------------------------
TRANSFER_DROP_LABELS = ["IUI", "TIC", "STIM", "RET"]

# Cycles to exclude
transfer_drop_cycles = set(df.loc[labels.isin(TRANSFER_DROP_LABELS), ID_COL])

# Cycles that contain ET or FET
transfer_cycles = set(df.loc[labels.isin(["ET", "FET"]), ID_COL])

# Keep transfer cycles that are not excluded
transfer_keep_ids = transfer_cycles - transfer_drop_cycles

transfers = df[df[ID_COL].isin(transfer_keep_ids)].copy()


In [44]:
print(retrievals.head())
print(retrievals.shape)

                                      id  \
2   f1806802-e10e-48db-98aa-18ec366fff78   
3   b0696bdb-5965-419d-813e-fe4e1c9f18f9   
8   d6e767b0-eaa4-4aae-9507-8fbcd9c0b8a4   
9   2e89732b-e123-493e-815f-4ea1549c94af   
16  af121ae4-a12d-4845-bcff-28208e234e35   

                                 cycleid label  daynum   eventdate  
2   c6e70948-17fb-4594-a3c3-657bff9f9557  STIM       1  2023-10-05  
3   c6e70948-17fb-4594-a3c3-657bff9f9557   LMP     -32  2023-09-03  
8   e11a96de-eb7e-480b-8c81-69c8b7be4e70  STIM       1  2023-05-19  
9   ce67cd1a-d9a4-448c-9d14-9d784683fa27  STIM       1  2023-06-16  
16  0554fe39-755c-4a1f-93cc-1ce7863c492e  STIM       1  2023-03-11  
(11108, 5)


In [45]:
print(transfers.head())
print(transfers.shape)

                                      id  \
25  bb693d57-eee7-4e40-8917-3d458bb5e22a   
26  3b5a708e-8852-4e60-a0ce-da847c729567   
51  699d595d-543b-4cb5-af9d-57036e9e3070   
52  fc4ada33-569f-4c66-9758-500d420a0c46   
53  e69f2807-fe2f-4e50-a73c-e3ff80796a15   

                                 cycleid label  daynum   eventdate  
25  14b828fe-de18-43a5-a8e8-27036915ba15   LMP       1  2023-06-24  
26  14b828fe-de18-43a5-a8e8-27036915ba15   FET      21  2023-07-14  
51  099a7025-2adc-4fa8-8587-90ac3987cb44   LMP       1  2023-08-22  
52  099a7025-2adc-4fa8-8587-90ac3987cb44   +LH      13  2023-09-03  
53  099a7025-2adc-4fa8-8587-90ac3987cb44   FET      19  2023-09-09  
(3764, 5)


In [53]:
# Relabelling some of the labels that are clearly belonging to other categories upon inspection of the data
# relabel "Day 11 TRG" as "TRG"
retrievals[LABEL_COL] = retrievals[LABEL_COL].str.strip().str.upper().replace({"DAY 11 TRG": "TRG"})

# relabel "DAY 8 TRG" as "TRG"
retrievals[LABEL_COL] = retrievals[LABEL_COL].str.strip().str.upper().replace({"DAY 8 TRG": "TRG"})

# relabel "DAY 13 DAY" as "TRG"
retrievals[LABEL_COL] = retrievals[LABEL_COL].str.strip().str.upper().replace({"DAY 13 DAY": "TRG"})

# relabel "DAY 12 DAY" as "TRG"
retrievals[LABEL_COL] = retrievals[LABEL_COL].str.strip().str.upper().replace({"DAY 12 DAY": "TRG"})

# relabel "DAY 12 RET" as "RET"
retrievals[LABEL_COL] = retrievals[LABEL_COL].str.strip().str.upper().replace({"DAY 12 RET": "RET"})


In [54]:
# now a count of each label
print(retrievals[LABEL_COL].value_counts())

label
STIM    3584
TRG     3115
RET     3025
LMP      940
+LH      362
LH        47
XX        32
CC         2
AFC        1
Name: count, dtype: int64


In [55]:
print(transfers[LABEL_COL].value_counts())

label
LMP    1541
FET    1454
TRG     322
+LH     272
ET      147
TEG      13
LH       11
PG        3
XX        1
Name: count, dtype: int64


In [52]:
# Investigating rare labels in retrievals
# Setting a threshold for what counts as a "rare" label
THRESHOLD = 10

counts = retrievals[LABEL_COL].value_counts()
rare_labels = counts[counts <= THRESHOLD].index

# cycleids that have at least one rare-label row
rare_cycle_ids = retrievals.loc[retrievals[LABEL_COL].isin(rare_labels), ID_COL].unique()

# ALL rows belonging to those cycles
rare_cycle_rows = retrievals[retrievals[ID_COL].isin(rare_cycle_ids)]

print(f"{len(rare_cycle_ids)} cycles contain a rare label")
print(f"{len(rare_cycle_rows)} rows total across those cycles")
rare_cycle_rows.sort_values([ID_COL])

print(rare_cycle_rows.head(60))

4 cycles contain a rare label
10 rows total across those cycles
                                         id  \
248    b260c953-228e-41c4-ac45-4d71c3fcf917   
249    23883f6d-916a-4e9d-8b18-53f91b3d9054   
250    1dd2e17d-af62-459d-a2a9-685ed0a604fa   
254    25cb61c0-bdff-4973-8d10-9a88ac43c6c3   
255    e0b19b2b-e367-42ef-806c-69f462ac72ce   
6971   5eaaea22-d291-42fb-ad83-0c5ce866a12d   
6972   0bc3b053-eec9-4be4-aff3-dd4b91aad8f7   
6973   51013d6d-d14e-4a65-9741-2f0fcf8da30e   
11580  e56b2bdb-eef2-4b49-a12e-30785e947c0e   
11581  ffcc30c9-efb9-4790-b537-c98cc1e26360   

                                    cycleid       label  daynum   eventdate  
248    a9822afe-3134-401f-89dd-81f72ffd6c59        STIM       1  2021-10-01  
249    a9822afe-3134-401f-89dd-81f72ffd6c59         AFC       5  2021-10-05  
250    a9822afe-3134-401f-89dd-81f72ffd6c59         TRG       9  2021-10-09  
254    e76312f6-3a3b-43b6-8685-347dc254140a        STIM       1  2021-11-19  
255    e76312f6-3a3b-43b6-86

In [57]:
# Investigating rare labels in transfers
# Setting a threshold for what counts as a "rare" label
THRESHOLD = 10

counts = transfers[LABEL_COL].value_counts()
rare_labels = counts[counts <= THRESHOLD].index

# cycleids that have at least one rare-label row
rare_cycle_ids = transfers.loc[transfers[LABEL_COL].isin(rare_labels), ID_COL].unique()

# ALL rows belonging to those cycles
rare_cycle_rows = transfers[transfers[ID_COL].isin(rare_cycle_ids)]

print(f"{len(rare_cycle_ids)} cycles contain a rare label")
print(f"{len(rare_cycle_rows)} rows total across those cycles")
rare_cycle_rows.sort_values([ID_COL])

print(rare_cycle_rows.head(60))

4 cycles contain a rare label
12 rows total across those cycles
                                        id  \
3176  3c0a7109-9407-40e4-ae05-d406d1e6563a   
3177  631b389c-73b9-45b0-92dd-1602bd056e55   
3178  31dd9bcf-8923-42ce-af31-8e56f9f9733a   
4347  014323e3-a96b-4177-bcfc-5e233ac5b608   
4348  99fc9fa0-d993-47bb-b087-cb52b6a2bb60   
4349  03da783b-2aed-4331-953c-c0ffcb832ea3   
5253  1e12b271-5017-40cf-b5dd-8c1dbf909e9f   
5254  f4f7c513-5dd8-4c12-8907-9fc63b79317f   
5255  af9520fa-f3fe-495f-bc3f-abd1d129a3a6   
6685  28bcc8ed-ffa4-47f3-9923-533eb26b254f   
6686  ffc6f540-4e64-465d-a979-3d6486a9cc6b   
6687  a5468482-f402-435a-a9aa-48cb06d7cbab   

                                   cycleid label  daynum   eventdate  
3176  6d300d8f-6793-43ec-baac-b60d1d816182   LMP       1  2022-03-19  
3177  6d300d8f-6793-43ec-baac-b60d1d816182   FET      20  2022-04-07  
3178  6d300d8f-6793-43ec-baac-b60d1d816182    XX     609  2023-11-17  
4347  6219b70e-0891-431d-b1ec-8961f4efaba0   LMP     

## 3. Patient profiling

Finding People who have STIM and TRG but not RET

In [59]:
labels = retrievals[LABEL_COL].astype(str).str.strip().str.upper()

# does each cycle contain STIM / TRG / RET anywhere in its rows?
flags = pd.DataFrame({
    ID_COL:    retrievals[ID_COL],
    "is_stim": labels == "STIM",
    "is_trg":  labels == "TRG",
    "is_ret":  labels == "RET",
})
per_cycle = flags.groupby(ID_COL)[["is_stim", "is_trg", "is_ret"]].any()

# classify each cycle (only STIM cycles get a label; others stay blank)
per_cycle["response_type"] = pd.NA
stim = per_cycle["is_stim"]
per_cycle.loc[stim &  per_cycle["is_ret"],                       "response_type"] = "normal"
per_cycle.loc[stim &  per_cycle["is_trg"] & ~per_cycle["is_ret"], "response_type"] = "?"
per_cycle.loc[stim & ~per_cycle["is_trg"] & ~per_cycle["is_ret"], "response_type"] = "poor"

# map the per-cycle label back onto every row of that cycle
retrievals["response_type"] = retrievals[ID_COL].map(per_cycle["response_type"])

In [61]:
# keeping only unique rows of cycleid, label where label == "STIM", eventdate, and response_type (dropping the other variables) and saving as response_df
response_df = retrievals.loc[df[LABEL_COL] == "STIM", [ID_COL, LABEL_COL, "eventdate", "response_type"]].drop_duplicates()

#display(response_df.head(20))

#counting the numer of each response type
print(response_df["response_type"].value_counts())


response_type
normal    3027
poor       428
?          129
Name: count, dtype: int64


In [62]:
#now isolating the cycleid's of the ? response type
questionable_cycles = response_df.loc[response_df["response_type"] == "?", ID_COL].unique()
print(questionable_cycles)

<StringArray>
['9c28dfce-7544-419f-bde3-3e8925890336',
 'a9822afe-3134-401f-89dd-81f72ffd6c59',
 '3beca621-0731-43f5-97ff-6bf34d1c0bee',
 'cb1cb192-b43c-4f63-87cf-496410e0d002',
 '7c24ce3e-496d-497e-9728-978dc59394b1',
 'e45795ca-7c37-40d4-9cee-30e24e23a8a8',
 '4497cb2e-104d-49ec-8f5e-223a91552265',
 '06265885-b8e5-41ce-bbca-24cf4927f379',
 '75903ad2-8a17-461b-82d2-e4f25aa07ff0',
 'b06d2a89-f62b-4dad-b15c-ebf389140701',
 ...
 '0724165b-8dba-43db-bd08-cf1e4c48056a',
 'a5c2df6c-dc88-41ef-9020-08fa156e5d0f',
 '0227c828-0267-47a1-b098-6f51a8186d11',
 '9a62070b-5b32-4a7d-a0d0-35ffa3343c8c',
 'ce581496-cf3f-4441-ae57-6281aba088ed',
 '979cb06c-22cd-4c0c-b7d5-26d9af19bece',
 '85083fea-252d-40cf-8601-910f9cf4b4ce',
 'bbcf7698-e1b8-400e-aac7-3c1d304025a8',
 '071dc18a-388e-42e7-8e90-6c419c19f520',
 'd8793ab4-d790-483d-84dc-b203c34f2ddf']
Length: 129, dtype: str


## 4. Getting Retrieval Date and Tranfer Date

Want to put this in wide format so that patients have a retrieval date associated with their stim cycles

In [74]:
# Getting Retrieval Dates for the Retrievals

# now isolating the cases that have label == "RET"
df_ret = retrievals.loc[retrievals[LABEL_COL] == "RET", [ID_COL, LABEL_COL, "eventdate"]].drop_duplicates()

# renaming column "retrieval_date" to override eventdate of the df_ret
df_ret = df_ret.rename(columns={"eventdate": "retrieval_date"})

#dropping label and response_type columns from df_ret
df_ret = df_ret.drop(columns=[LABEL_COL])

# now combining df_ret with response_type by cycleid
df_ret_combined = pd.merge(response_df, df_ret, on=ID_COL, how="left")

# renaming column "stim_start" to override eventdate of the df_combined
df_ret_combined = df_ret_combined.rename(columns={"eventdate": "stim_start"})

# now isolating the cases that have label == "TRG"
df_trg = retrievals.loc[retrievals[LABEL_COL] == "TRG", [ID_COL, LABEL_COL, "eventdate"]].drop_duplicates()

# renaming column "trigger_date" to override eventdate of the df_ret
df_trg = df_trg.rename(columns={"eventdate": "trigger_date"})

#dropping label and response_type columns from df_ret
df_trg = df_trg.drop(columns=[LABEL_COL])

# now combining df_ret with response_type by cycleid
df_ret_combined = pd.merge(df_ret_combined, df_trg, on=ID_COL, how="left")

# displaying the first 20 rows of the combined dataframe
display(df_ret_combined.head(20))
df_ret_combined.shape

,cycleid,label,stim_start,response_type,retrieval_date,trigger_date
0,c6e70948-17fb-4594-a3c3-657bff9f9557,STIM,2023-10-05,poor,NaN,NaN
1,e11a96de-eb7e-480b-8c81-69c8b7be4e70,STIM,2023-05-19,poor,NaN,NaN
2,ce67cd1a-d9a4-448c-9d14-9d784683fa27,STIM,2023-06-16,poor,NaN,NaN
3,0554fe39-755c-4a1f-93cc-1ce7863c492e,STIM,2023-03-11,normal,2023-03-24,2023-03-22
4,b6cc920b-2cd4-4ea5-9aac-1bce596f78b7,STIM,2023-09-02,normal,2023-09-16,2023-09-14
5,482079e1-c78b-4e9a-bc4d-7b9c33fc6f18,STIM,2023-06-09,normal,2023-06-21,2023-06-19
6,03705db5-51dc-47a5-91a7-a21cec4d9530,STIM,2023-03-24,normal,2023-04-08,2023-04-06
7,4c9c143b-aba9-4be5-a70f-32b5fac4f8b4,STIM,2023-05-08,normal,2023-05-27,2023-05-25
8,2b309714-83fa-4a8f-b193-97cbb9ba2f39,STIM,2023-11-15,normal,2023-11-27,2023-11-25
9,0b6ae4b4-fbd4-4429-b86d-12c4f41c6456,STIM,2023-06-28,normal,2023-07-10,2023-07-08


(3591, 6)

In [ ]:
# Getting Transfer Dates for the Transfers

# now isolating the cases that have label == "FET" or ET
df_tr = transfers.loc[transfers[LABEL_COL].isin(["FET", "ET"]), [ID_COL, LABEL_COL, "eventdate"]].drop_duplicates()

# renaming column "retrieval_date" to override eventdate of the df_ret
df_tr = df_tr.rename(columns={"eventdate": "transfer_date"})

#dropping label and response_type columns from df_tr
df_tr = df_tr.drop(columns=[LABEL_COL])

## now combining df_tr with LMP by cycleid
df_lmp = transfers.loc[transfers[LABEL_COL] == "LMP", [ID_COL, LABEL_COL, "eventdate"]].drop_duplicates()
df_lmp = df_lmp.rename(columns={"eventdate": "last_menstrual_period"})
df_lmp = df_lmp.drop(columns=[LABEL_COL])

# now merging transfer data
df_tr_combined = pd.merge(df_tr, df_lmp, on=ID_COL, how="left")


,cycleid,transfer_date,last_menstrual_period
0,14b828fe-de18-43a5-a8e8-27036915ba15,2023-07-14,2023-06-24
1,099a7025-2adc-4fa8-8587-90ac3987cb44,2023-09-09,2023-08-22
2,5eb5fe49-c5e6-4c10-80bd-2d876f111d5c,2023-11-22,2023-11-09
3,689dedc7-90c8-4b5d-ad46-f6fde7272cb5,2023-10-30,2023-10-15
4,73a353d7-16f9-4840-91d1-7f4bb1002216,2023-10-24,2023-10-08
5,ce0937d7-4aa8-4a3d-891e-ff8562c77bc8,2023-07-19,2023-07-05
6,5f7b91cd-64bc-4571-85f2-76ea8e6503f6,2023-07-15,2023-06-26
7,d028ae2c-bd7e-4340-9bb1-8438efdd7630,2024-04-15,2024-03-22
8,a42e6863-c241-455d-b2fa-3042d44e0d19,2023-10-20,2023-10-04
9,080df659-479a-44dd-bccd-881596208f09,2023-12-06,2023-11-19


## 5. Saving to csv

Want to save both dfs to use alongside other tables

In [ ]:
# save as csv
df_ret_combined.to_csv("../data/retrieval_cycles.csv", index=False)

df_tr_combined.to_csv("../data/transfer_cycles.csv", index=False)